# Лаборатори 3 CSP хэрэгжүүлэлт

Зорилго: CSP-ийг ерөнхий бүтцээр загварчилж, backtracking, MRV, degree, LCV-г хэрэгжүүлэх ба зангилааны тоогоор харьцуулах.

Эх сурвалж: **Lab3_CSP_Manual.pdf**, 3.1–3.12 хэсэг. Notebook нь бүх алхам, ажиллуулсан шалгалт, нэмэлт даалгавар, хэмжилтийг агуулна. Слайд хавсаргаагүй тул зорилго болон онолыг гарын авлагын хүрээнд тайлбарлав.

Зааврын 7-р хуудсанд хүснэгтийн DM-г дарааллын дүрэмд DB гэж бичсэн. Энд **AI < DM** гэж үзэв. 1-р бүлэгт эхэндээ гурван хичээл байгаа тул нэг хичээл нэмэхэд дөрөв, хоёр нэмэхэд тав болно. Хоёр хувилбарыг тусад нь шалгана.

## Алхам 1 CSP класс

Оноолтыг dictionary-аар дүрслэнэ. `is_consistent` нь зөвхөн оноогдсон хөршүүдийг шалгана. `nodes` нь зөрчилгүй утгыг оноолтод нэмсэн тоо (заавар, алхам 3). `attempts` нь хайлтын давталтаар туршсан утгын тоо; heuristic доторх үнэлгээ үүнд орохгүй.

In [1]:
class CSP:
    # Хувьсагч, домэйн, хөрш, хязгаарлалт болон тоолуурыг хадгална.
    def __init__(self, variables, domains, neighbors, constraint):
        self.variables = list(variables)
        self.domains = {v: list(domains[v]) for v in self.variables}
        self.neighbors = {v: list(neighbors[v]) for v in self.variables}
        self.constraint = constraint
        self.nodes = 0
        self.attempts = 0

    # Зөвхөн оноогдсон хөршүүдтэй зөрчилдөх эсэхийг шалгана.
    def is_consistent(self, var, value, assignment):
        return all(
            self.constraint(var, value, other, assignment[other])
            for other in self.neighbors[var] if other in assignment
        )

    # Анхны дарааллаар оноогдоогүй хувьсагчдыг буцаана.
    def unassigned(self, assignment):
        return [v for v in self.variables if v not in assignment]


class SearchLimitExceeded(RuntimeError):
    """Зөвшөөрөгдсөн зангилааны хязгаарт хүрсэн хайлт."""

In [2]:
# Гурван хувьсагчтай жижиг CSP-ийн зөрчилтэй болон зөрчилгүй оноолтыг шалгана.
tiny = CSP(['A', 'B', 'C'], {v: [1, 2] for v in ['A', 'B', 'C']},
           {'A': ['B'], 'B': ['A'], 'C': []}, lambda A, a, B, b: a != b)
assert not tiny.is_consistent('B', 1, {'A': 1})
assert tiny.is_consistent('B', 2, {'A': 1})
assert tiny.is_consistent('C', 1, {'A': 1})
assert tiny.unassigned({'A': 1}) == ['B', 'C']
print('Зөрчилтэй: False; зөрчилгүй: True; оноогдоогүй: B, C')

Зөрчилтэй: False; зөрчилгүй: True; оноогдоогүй: B, C


## Алхам 2 Австралийн газрын зураг

Хөрш мужууд өөр өнгөтэй байна. Ирмэг бүрийг хоёр чиглэлд бүртгэнэ. Tasmania буюу T нь хөршгүй.

In [3]:
# Хоёр чиглэлтэй хөршийн жагсаалтыг ирмэгүүдээс үүсгэнэ.
def make_neighbors(variables, edges):
    neighbors = {v: [] for v in variables}
    for a, b in edges:
        neighbors[a].append(b)
        neighbors[b].append(a)
    return neighbors


# Австралийн мужуудыг өгсөн өнгөөр будах CSP үүсгэнэ.
def make_map_csp(colors=("red", "green", "blue")):
    variables = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
    edges = [("WA", "NT"), ("WA", "SA"), ("NT", "SA"),
             ("NT", "Q"), ("SA", "Q"), ("SA", "NSW"),
             ("SA", "V"), ("Q", "NSW"), ("NSW", "V")]
    # Хөрш мужууд ялгаатай өнгөтэй байх ёстой.
    def constraint(a, color_a, b, color_b):
        return color_a != color_b
    return CSP(variables, {v: list(colors) for v in variables},
               make_neighbors(variables, edges), constraint)

In [4]:
map_csp = make_map_csp()
assert [len(map_csp.neighbors[v]) for v in ['SA', 'T', 'WA']] == [5, 0, 2]
print({v: len(map_csp.neighbors[v]) for v in ['SA', 'T', 'WA']})

{'SA': 5, 'T': 0, 'WA': 2}


## Алхам 3 Backtracking хайлт

Эхлээд жагсаалтын эхний хувьсагч болон домэйны анхны дарааллыг хэрэглэнэ. Буцахдаа оноолтыг устгаж, шийдийг copy хийнэ. `check_solution` нь домэйн, бүрэн оноолт, бүх хөршийн хязгаарлалтыг тусад нь шалгана.

In [5]:
# Heuristic-гүй үед жагсаалтын эхний оноогдоогүй хувьсагчийг сонгоно.
def select_unassigned_variable(csp, assignment, use_mrv=False, use_degree=False):
    return csp.unassigned(assignment)[0]

# Heuristic-гүй үед домэйны анхны дарааллыг хадгална.
def order_domain_values(csp, var, assignment, use_lcv=False):
    return list(csp.domains[var])

In [6]:
# Consistent оноолтын тоог өсгөхөөс өмнө хайлтын хязгаарыг шалгана.
def record_node(csp, node_limit):
    if node_limit is not None and csp.nodes >= node_limit:
        raise SearchLimitExceeded(f"Зангилааны хязгаар: {node_limit}")
    csp.nodes += 1


# Рекурсив backtracking нь эхний шийд эсвэл None буцаана.
def backtrack(csp, assignment, use_mrv=False, use_degree=False,
              use_lcv=False, node_limit=None):
    if len(assignment) == len(csp.variables):
        return assignment.copy()
    var = select_unassigned_variable(csp, assignment, use_mrv, use_degree)
    for value in order_domain_values(csp, var, assignment, use_lcv):
        csp.attempts += 1
        if csp.is_consistent(var, value, assignment):
            record_node(csp, node_limit)
            assignment[var] = value
            try:
                result = backtrack(csp, assignment, use_mrv, use_degree,
                                   use_lcv, node_limit)
            finally:
                del assignment[var]
            if result is not None:
                return result
    return None


# Хайлт бүрийн өмнө тоолуурыг тэглэж хоосон оноолтоос эхлүүлнэ.
def backtracking_search(csp, use_mrv=False, use_degree=False,
                        use_lcv=False, node_limit=None):
    csp.nodes = csp.attempts = 0
    return backtrack(csp, {}, use_mrv, use_degree, use_lcv, node_limit)


# Шийдийг хайлтаас үл хамааран бүрэн байдал, домэйн, хязгаарлалтаар шалгана.
def check_solution(csp, assignment):
    if assignment is None or set(assignment) != set(csp.variables):
        return False
    if any(assignment[v] not in csp.domains[v] for v in csp.variables):
        return False
    return all(csp.constraint(v, assignment[v], w, assignment[w])
               for v in csp.variables for w in csp.neighbors[v])

In [7]:
map_solution = backtracking_search(map_csp)
assert check_solution(map_csp, map_solution)
print('Шийд:', map_solution)
print('Зөв:', check_solution(map_csp, map_solution), 'Зангилаа:', map_csp.nodes)
assert backtracking_search(map_csp) == map_solution
assert map_csp.nodes == 7
print('Дахин хайхад тоолуур тэглэгдсэн:', map_csp.nodes)

Шийд: {'WA': 'red', 'NT': 'green', 'SA': 'blue', 'Q': 'red', 'NSW': 'green', 'V': 'red', 'T': 'red'}
Зөв: True Зангилаа: 7
Дахин хайхад тоолуур тэглэгдсэн: 7


## Алхам 4 N Queens загвар ба бүх шийд

Хувьсагч нь мөр, утга нь багана. Хоёр хатны багана ялгаатай, баганын зөрүү мөрийн зөрүүтэй тэнцүү биш байна. Өмнөх solver-ийг өөрчлөхгүй хэрэглэнэ. Бүх шийдийг тоолох хувилбар шийд олсон үед хайлтыг үргэлжлүүлнэ.

In [8]:
# Мөрийг хувьсагч, баганыг домэйн болгон N-Queens CSP үүсгэнэ.
def make_queens_csp(n):
    if n < 1:
        raise ValueError("CSP самбарын n нь 1-ээс багагүй байна")
    variables = list(range(n))
    # Хатад нэг багана болон нэг диагональд байрлахгүй.
    def constraint(row_a, col_a, row_b, col_b):
        return col_a != col_b and abs(col_a - col_b) != abs(row_a - row_b)
    return CSP(variables, {v: list(range(n)) for v in variables},
               {v: [w for w in variables if w != v] for v in variables},
               constraint)


# Бүх шийдийг тоолох эсвэл тусдаа dictionary болгон цуглуулах DFS хийнэ.
def enumerate_solutions(csp, use_mrv=False, use_degree=False,
                        use_lcv=False, collect=True, node_limit=None):
    csp.nodes = csp.attempts = 0
    assignment, solutions = {}, []
    # Шийд олсон ч бусад салааг үргэлжлүүлнэ.
    def visit():
        if len(assignment) == len(csp.variables):
            if collect:
                solutions.append(assignment.copy())
            return 1
        total = 0
        var = select_unassigned_variable(csp, assignment, use_mrv, use_degree)
        for value in order_domain_values(csp, var, assignment, use_lcv):
            csp.attempts += 1
            if csp.is_consistent(var, value, assignment):
                record_node(csp, node_limit)
                assignment[var] = value
                try:
                    total += visit()
                finally:
                    del assignment[var]
        return total
    count = visit()
    return solutions if collect else count


# Шийдүүдийг хадгалахгүйгээр зөвхөн тоог буцаана.
def count_solutions(csp, use_mrv=False, use_degree=False, use_lcv=False):
    return enumerate_solutions(csp, use_mrv, use_degree, use_lcv, collect=False)


# Хатан байрласан нүдийг Q, бусдыг цэгээр хэвлэнэ.
def print_board(assignment, n):
    for row in range(n):
        print(" ".join("Q" if assignment[row] == col else "." for col in range(n)))

In [9]:
expected_counts = [1, 0, 0, 2, 10, 4, 40, 92]
counts = [count_solutions(make_queens_csp(n)) for n in range(1, 9)]
assert counts == expected_counts
print('N          :', list(range(1, 9)))
print('Шийдийн тоо:', counts)
four = enumerate_solutions(make_queens_csp(4))
for s in four:
    assert check_solution(make_queens_csp(4), s)
    print('\nШийд:', [s[r] for r in range(4)])
    print_board(s, 4)

N          : [1, 2, 3, 4, 5, 6, 7, 8]
Шийдийн тоо: [1, 0, 0, 2, 10, 4, 40, 92]

Шийд: [1, 3, 0, 2]
. Q . .
. . . Q
Q . . .
. . Q .

Шийд: [2, 0, 3, 1]
. . Q .
Q . . .
. . . Q
. Q . .


### Өмнөх DFS N Queens даалгаврын шалгалт

`dfs_n_queens(n)` нь яг нэг параметртэй; n < 1 үед хоосон жагсаалт буцаана. Багануудыг өсөх дарааллаар шалгах тул хүлээгдсэн шийдийн дарааллыг хадгална.

In [10]:
# Мөр бүрд хатан байрлуулж, зөрчилгүй бүх шийдийг DFS-ээр олно.
def dfs_n_queens(n):
    """Return all N-Queens solutions in ascending column DFS order."""
    if n < 1:
        return []

    solutions = []
    placement = []
    columns = set()
    diagonals_down = set()
    diagonals_up = set()

    def dfs(row):
        if row == n:
            solutions.append(placement.copy())
            return

        for col in range(n):
            down = row - col
            up = row + col
            if col in columns or down in diagonals_down or up in diagonals_up:
                continue

            placement.append(col)
            columns.add(col)
            diagonals_down.add(down)
            diagonals_up.add(up)
            dfs(row + 1)
            placement.pop()
            columns.remove(col)
            diagonals_down.remove(down)
            diagonals_up.remove(up)

    dfs(0)
    return solutions


In [11]:
import inspect
assert len(inspect.signature(dfs_n_queens).parameters) == 1
for n in [-5, -1, 0, 2, 3]:
    assert dfs_n_queens(n) == []
assert dfs_n_queens(1) == [[0]]
assert dfs_n_queens(4) == [[1, 3, 0, 2], [2, 0, 3, 1]]
assert dfs_n_queens(5) == [[0,2,4,1,3],[0,3,1,4,2],[1,3,0,2,4],
    [1,4,2,0,3],[2,0,3,1,4],[2,4,1,3,0],[3,0,2,4,1],
    [3,1,4,2,0],[4,1,3,0,2],[4,2,0,3,1]]
assert len(dfs_n_queens(5)) == 10
assert len(dfs_n_queens(8)) == 92
for n in range(1, 9):
    generic = enumerate_solutions(make_queens_csp(n))
    assert dfs_n_queens(n) == [[s[r] for r in range(n)] for s in generic]
print('DFS даалгаврын бүх шаардлага болон ерөнхий solver-той харьцуулалт: OK')

DFS даалгаврын бүх шаардлага болон ерөнхий solver-той харьцуулалт: OK


## Алхам 5 MRV

Одоогийн оноолтод зөрчилгүй үлдсэн утга хамгийн цөөн хувьсагчийг сонгоно. Тэнцвэл анхны дарааллыг хадгална.

In [12]:
# Одоогийн оноолттой зөрчилгүй домэйны утгуудыг олно.
def legal_values(csp, var, assignment):
    return [value for value in csp.domains[var]
            if csp.is_consistent(var, value, assignment)]

In [13]:
# Одоогийн боломжит утгын тоогоор MRV хувьсагчийг сонгоно.
def select_unassigned_variable(csp, assignment, use_mrv=True, use_degree=False):
    remaining = csp.unassigned(assignment)
    if not use_mrv:
        return remaining[0]
    return min(remaining, key=lambda v: len(legal_values(csp, v, assignment)))

In [14]:
assert select_unassigned_variable(make_map_csp(), {'WA': 'red'}) == 'NT'
assert count_solutions(make_queens_csp(8), use_mrv=True) == 92
print('WA=red дараа MRV: NT; MRV-тэй 8-Queens: 92 шийд')

WA=red дараа MRV: NT; MRV-тэй 8-Queens: 92 шийд


## Алхам 6 Degree

MRV тэнцсэн үед оноогдоогүй хөрш хамгийн олон хувьсагчийг сонгоно. Дахин тэнцвэл анхны дарааллыг хадгална.

In [15]:
# MRV ашиглан сонгоод тэнцвэл оноогдоогүй хөршийн degree-г хэрэглэнэ.
def select_unassigned_variable(csp, assignment, use_mrv=True, use_degree=False):
    remaining = csp.unassigned(assignment)
    # Эрэмбэлэх түлхүүр; min нь тэнцсэн үед анхны дарааллыг хадгална.
    def key(var):
        mrv = len(legal_values(csp, var, assignment)) if use_mrv else 0
        degree = (sum(w not in assignment for w in csp.neighbors[var])
                  if use_degree else 0)
        return mrv, -degree
    return min(remaining, key=key)

In [16]:
assert select_unassigned_variable(make_map_csp(), {}, True, True) == 'SA'
print('Хоосон оноолтод MRV + Degree: SA, 5 хөрш')

Хоосон оноолтод MRV + Degree: SA, 5 хөрш


## Алхам 7 LCV

Оноогдоогүй хөршүүдийн одоогоор боломжтой утгаас шинээр хасах тоог тооцож өсөхөөр эрэмбэлнэ. Оноолт болон домэйныг өөрчлөхгүй.

In [17]:
# Оноогдоогүй хөршүүдээс хамгийн цөөн боломжит утга хасах утгыг эхэлж өгнө.
def order_domain_values(csp, var, assignment, use_lcv=True):
    if not use_lcv:
        return list(csp.domains[var])
    values = legal_values(csp, var, assignment)
    neighbor_values = {
        w: legal_values(csp, w, assignment)
        for w in csp.neighbors[var] if w not in assignment
    }
    # Одоогоор боломжтой утгуудаас шинээр хасагдах тоог тоолно.
    def eliminated(value):
        return sum(not csp.constraint(var, value, w, other_value)
                   for w, choices in neighbor_values.items()
                   for other_value in choices)
    return sorted(values, key=eliminated)

In [18]:
partial = {'WA': 'red', 'NT': 'green'}
before = partial.copy()
ordered = order_domain_values(make_map_csp(), 'Q', partial)
assert ordered == ['red', 'blue'] and partial == before
print('Q-ийн LCV дараалал:', ordered, '; оноолт өөрчлөгдөөгүй:', partial == before)
for mrv, degree, lcv in [(False, False, False), (True, False, False),
                          (True, True, False), (True, True, True)]:
    csp = make_queens_csp(8)
    all_solutions = enumerate_solutions(csp, mrv, degree, lcv)
    assert len(all_solutions) == 92
    assert len({tuple(s[r] for r in range(8)) for s in all_solutions}) == 92
    assert all(check_solution(csp, s) for s in all_solutions)
print('Дөрвөн тохиргоо тус бүр: 92 давхцаагүй зөв шийд')

Q-ийн LCV дараалал: ['red', 'blue'] ; оноолт өөрчлөгдөөгүй: True


Дөрвөн тохиргоо тус бүр: 92 давхцаагүй зөв шийд


## Алхам 8 Харьцуулалтын туршилт

Ижил хувьсагч, домэйн, tie-break дарааллаар эхний шийдийг олно. Туршилт бүрд шинэ CSP үүсгэж, тоолуурыг тэглэнэ. Тохиргоо бүрийг гурван удаа ажиллуулж хугацааны медианыг авна. Хугацаанд heuristic-ийн тооцоолол багтана. Зангилааны хязгаар 1 000 000; хязгаарт хүрвэл хайлтыг тасална. Бүх шийдийг тусдаа шалгагчаар баталгаажуулна.

In [19]:
CONFIGS = [
    ("A", "Heuristic-гүй", False, False, False),
    ("B", "MRV", True, False, False),
    ("C", "MRV + Degree", True, True, False),
    ("D", "MRV + Degree + LCV", True, True, True),
]


# Ижил нөхцөлтэй эхний шийдийн хайлтыг хэмжиж бодит үр дүнг буцаана.
def benchmark(node_limit=1_000_000, repeats=3):
    from time import perf_counter
    from statistics import median
    rows = []
    for code, label, mrv, degree, lcv in CONFIGS:
        for name, n in [("Map", None), ("8-Queens", 8),
                        ("12-Queens", 12), ("16-Queens", 16)]:
            durations = []
            for _ in range(repeats):
                csp = make_map_csp() if n is None else make_queens_csp(n)
                start = perf_counter()
                try:
                    result = backtracking_search(csp, mrv, degree, lcv, node_limit)
                    status = "solved" if result is not None else "unsatisfiable"
                except SearchLimitExceeded:
                    result, status = None, "limit"
                durations.append((perf_counter() - start) * 1000)
                if status == "solved" and not check_solution(csp, result):
                    raise AssertionError("Хүчингүй шийд")
            rows.append(dict(config=code, label=label, problem=name,
                             nodes=csp.nodes, attempts=csp.attempts,
                             milliseconds=median(durations), status=status,
                             solution=result, repeats=repeats))
    return rows

In [20]:
import json
from pathlib import Path
import platform
rows = benchmark(repeats=3)
Path('tmp/lab3').mkdir(parents=True, exist_ok=True)
Path('tmp/lab3/benchmark.json').write_text(
    json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
print('Python:', platform.python_version(), '; давталт: 3; хугацаа: медиан мс')
print(f"{'Тохиргоо':<10} {'Бодлого':<12} {'Зангилаа':>9} {'Оролдлого':>10} {'мс':>10}")
for r in rows:
    print(f"{r['config']:<10} {r['problem']:<12} {r['nodes']:>9} {r['attempts']:>10} {r['milliseconds']:>10.3f}")
assert all(r['status'] == 'solved' for r in rows)
best_code = min('ABCD', key=lambda c: sum(r['nodes'] for r in rows if r['config'] == c))
best_config = next(c for c in CONFIGS if c[0] == best_code)
print('Нийт зангилаагаар хамгийн бага тохиргоо:', best_config[1])

Python: 3.12.14 ; давталт: 3; хугацаа: медиан мс
Тохиргоо   Бодлого       Зангилаа  Оролдлого         мс
A          Map                  7         11      0.042
A          8-Queens           113        876      1.810
A          12-Queens          261       3066      7.357
A          16-Queens        10052     160712    457.439
B          Map                  7         11      0.118
B          8-Queens            75        572      5.473
B          12-Queens          153       1770     23.331
B          16-Queens           44        584     11.360
C          Map                  7         15      0.147
C          8-Queens            75        572      4.759
C          12-Queens          153       1770     23.038
C          16-Queens           44        584     12.002
D          Map                  7          7      0.224
D          8-Queens            39         39      4.034
D          12-Queens           32         32      9.253
D          16-Queens          332        332    139.625

## Нэмэлт даалгавар Цагийн хуваарь

Багш B-ийн боломжтой слотыг домэйноор [1, 2] болгон хязгаарлана. Нэг багш эсвэл нэг бүлэгтэй хичээлүүд давхцахгүй. Зааврын нэрийн зөрүүг AI < DM гэж тайлбарласан.

In [21]:
# Сургалтын цагийн хуваарьт багш, бүлэг, дарааллын binary хязгаарлалт үүсгэнэ.
def make_schedule_csp(extra_group1=0):
    variables = ["AI", "DM", "NET", "OS", "MATH", "ENG"]
    teachers = dict(zip(variables, ["A", "A", "B", "B", "C", "C"]))
    groups = dict(zip(variables, [1, 2, 1, 2, 2, 1]))
    for i in range(extra_group1):
        name = f"EXTRA{i + 1}"
        variables.append(name)
        teachers[name], groups[name] = f"D{i + 1}", 1
    domains = {v: [1, 2] if teachers[v] == "B" else [1, 2, 3, 4]
               for v in variables}
    edges = [(a, b) for i, a in enumerate(variables) for b in variables[i + 1:]
             if teachers[a] == teachers[b] or groups[a] == groups[b]
             or {a, b} == {"AI", "DM"}]
    # Зааврын DB-г хүснэгтэд байгаа DM гэж үзэж AI < DM шаардлагыг шалгана.
    def constraint(a, slot_a, b, slot_b):
        if (teachers[a] == teachers[b] or groups[a] == groups[b]) and slot_a == slot_b:
            return False
        if a == "AI" and b == "DM":
            return slot_a < slot_b
        if a == "DM" and b == "AI":
            return slot_b < slot_a
        return True
    csp = CSP(variables, domains, make_neighbors(variables, edges), constraint)
    csp.teachers, csp.groups = teachers, groups
    return csp

In [22]:
flags = best_config[2:]
schedule = make_schedule_csp()
schedule_solution = backtracking_search(schedule, *flags)
assert check_solution(schedule, schedule_solution)
print('Тохиргоо:', best_config[1], '; зангилаа:', schedule.nodes)
for slot in range(1, 5):
    courses = [v for v in schedule.variables if schedule_solution[v] == slot]
    print('Слот', slot, ':', ', '.join(courses))
one_extra = make_schedule_csp(1)
one_solution = backtracking_search(one_extra, *flags)
assert check_solution(one_extra, one_solution)
print('1 нэмэлт хичээл, 1-р бүлэг нийт 4:', one_solution)
two_extra = make_schedule_csp(2)
assert backtracking_search(two_extra, *flags) is None
print('2 нэмэлт хичээл, 1-р бүлэг нийт 5: None')
print('5 хичээлд ялгаатай 5 слот шаардлагатай боловч зөвхөн 4 слот байна.')
Path('tmp/lab3/schedule.json').write_text(json.dumps({
    'config': best_code, 'solution': schedule_solution, 'nodes': schedule.nodes,
    'one_extra_solution': one_solution, 'five_courses_result': None
}, ensure_ascii=False, indent=2), encoding='utf-8')

Тохиргоо: MRV ; зангилаа: 6
Слот 1 : NET, MATH
Слот 2 : AI, OS
Слот 3 : DM, ENG
Слот 4 : 
1 нэмэлт хичээл, 1-р бүлэг нийт 4: {'NET': 1, 'OS': 2, 'AI': 2, 'DM': 3, 'MATH': 1, 'ENG': 3, 'EXTRA1': 4}
2 нэмэлт хичээл, 1-р бүлэг нийт 5: None
5 хичээлд ялгаатай 5 слот шаардлагатай боловч зөвхөн 4 слот байна.


295

## Нэмэлт даалгавар Хоёр өнгийн газрын зураг

WA, NT, SA нь бие биетэйгээ хиллэдэг гурвалжин үүсгэнэ. Гурван харилцан хөршид гурван өөр өнгө шаардлагатай тул хоёр өнгө хүрэлцэхгүй.

In [23]:
two_colors = make_map_csp(('red', 'green'))
assert backtracking_search(two_colors, True, True, True) is None
print('Хоёр өнгийн газрын зураг: None; зангилаа:', two_colors.nodes)

Хоёр өнгийн газрын зураг: None; зангилаа: 4


## Хэлэлцүүлгийн асуултууд

1. **Minimax ба CSP.** Өмнөх TicTacToe код төлөвийг 9 нүдтэй жагсаалтаар хадгалж, тоглогчдын нүүдэл ба үнэлгээгээр хайсан. CSP нь variables, domains, neighbors, constraint ашиглах тул ижил solver-оор газрын зураг, хатад, хуваарийг шийдэж чадна.

2. **Хувьсагч ба утгын сонголт.** 16-Queens-д A-ийн 10 052 зангилаа MRV-тэй B-д 44 болсон. Харин C дээр LCV нэмсэн D нь 332 зангилаа болсон. Энэ туршилтад хувьсагчийн сонголт 16-Queens-д хамгийн их нөлөөлөв; 8 ба 12-Queens-д LCV нэмэлт хэмнэлт өгсөн.

3. **MRV fail-first.** Боломжит утгагүй хувьсагчийг эрт сонговол бусад хувьсагчид шаардлагагүй оноолт хийхээс өмнө тухайн салааг хаана.

4. **Хязгаарлалтын төрөл.** Багш B-ийн боломжтой слот нь unary; хоёр хичээлийн багш эсвэл бүлгийн давхцал болон AI < DM нь binary. Нэг бүлэг эсвэл багшийн бүх хичээлийг AllDifferent гэж нийлүүлж загварчилбал global болно; энэ хэрэгжүүлэлтэд хос бүрээр шалгасан.

## Алхам 8 Харьцуулалтын хүснэгт ба хариулт

| Тохиргоо | Газрын зураг | 8 Queens | 12 Queens | 16 Queens |
|---|---:|---:|---:|---:|
| A Heuristic-гүй | 7 | 113 | 261 | 10052 |
| B MRV | 7 | 75 | 153 | 44 |
| C MRV + Degree | 7 | 75 | 153 | 44 |
| D MRV + Degree + LCV | 7 | 39 | 32 | 332 |

**1. Аль тохиргоо хамгийн их хэмнэлт өгсөн бэ?** 8-Queens ба 12-Queens-д D; 16-Queens-д B болон C. Нийт зангилаа A=10433, B=C=279, D=410. Газрын зураг бүгд 7. Нийт зангилаагаар B/C тэнцсэн тул нэмэлт degree үнэлгээ шаарддаггүй B-г хуваарьт хэрэглэв. 16-Queens-д MRV 99.56% хэмнэсэн.

**2. N-Queens-д Degree яагаад өөрчлөлтгүй вэ?** Хөршийн граф бүрэн тул k оноогдоогүй мөр үлдвэл мөр бүрийн оноогдоогүй хөрш k−1 байна. Иймээс degree tie-break хийх ялгаа үүсэхгүй.

**3. LCV бүх тохиолдолд хурдасгасан уу?** Үгүй. 16-Queens-д C-ийн 44 зангилаа D-д 332 болсон. LCV нь ойрын хөршүүдийн утгыг хадгалах локал үнэлгээ тул эхний шийд рүү хамгийн богино замыг батлахгүй. Мөн heuristic тооцоолоход хугацаа зарцуулна. Хугацааг дээрх бодит хэмжилтийн хүснэгтээс харна. Зангилаа цөөрөх нь хугацаа заавал буурна гэсэн үг биш.